# 🎵 One-Shot Symbolic Music Style Transfer — v4
## Colab Notebook (Step-by-step)

### ⚠️ Rule Change (Professor's Note)
> **Y is NOT used during training** — this is a generative model task.  
> Scoring functions do not use Y. Only X (content) and Z (style) are inputs.

### Strategy
- **CP guaranteed**: output = X × velocity_scale → pitches never change → chroma preserved
- **SF maximised**: learn to match Z's velocity/rhythm distribution + copy Z's drum
- **Training loss** = CP loss + SF loss (directly optimise the evaluation metrics)

### Execution Flow
```
1. Run ALL cells top-to-bottom (Ctrl+F9)
2. Choose MODE A (immediate, no training) OR MODE B (trained, ~1hr)
3. Download /content/submission.csv and submit on Kaggle
```


## ✅ Step 1 — Install & Imports
Run once per session.

In [ ]:
# Install required packages
!pip install einops -q
print("Packages ready.")

In [ ]:
import os, zipfile, math, time, random, gc
from pathlib import Path
from copy import deepcopy
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from einops import rearrange

# Reproducibility
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

## ✅ Step 2 — Load Dataset
Upload `dataset.zip` to `/content/` first, then run this cell.


In [ ]:
ZIP_PATH  = "/content/dataset.zip"
DATA_ROOT = Path("/content/dataset")

if not DATA_ROOT.exists():
    print("Extracting dataset.zip ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall("/content")
    print("Done.")
else:
    print("Already extracted.")

MANIFEST_PATH = DATA_ROOT / "manifest.csv"
assert MANIFEST_PATH.exists(), f"manifest.csv not found at {MANIFEST_PATH}"

df_manifest = pd.read_csv(MANIFEST_PATH)
print(f"Items: {len(df_manifest)} | Splits: {df_manifest['split'].value_counts().to_dict()}")
df_manifest.head(3)

## ✅ Step 3 — Configuration
All hyperparameters in one place. Adjust if needed.


In [ ]:
class Config:
    # Paths
    DATA_ROOT = DATA_ROOT
    SAVE_DIR  = Path("/content/checkpoints")
    SUB_PATH  = Path("/content/submission.csv")

    # Piano roll (EDA: n_tracks=ALWAYS 4, drum=ALWAYS present)
    N_PITCH  = 128
    N_STEPS  = 128

    # Model
    HIDDEN   = 256   # increase to 384 if VRAM allows
    N_HEADS  = 8
    N_LAYERS = 4

    # Training (NO Y — self-supervised on X and Z only)
    BATCH_SIZE   = 16
    LR           = 1e-3
    WEIGHT_DECAY = 1e-4
    EPOCHS       = 60
    WARMUP_STEPS = 300
    EMA_DECAY    = 0.999
    GRAD_CLIP    = 1.0
    AMP          = True

    # Loss weights (directly optimise CP and SF evaluation metrics)
    W_CHROMA = 2.0   # → CP score  (chroma of output must match X)
    W_HIST   = 3.0   # → SF score  (distribution of output must match Z)
    W_DRUM   = 2.5   # → SF#4      (drum pattern must match Z's drum)

    # Inference
    BLEND_ALPHA    = 0.35  # for Mode A deterministic baseline
    INFER_BATCH    = 8
    NOTE_THRESHOLD = 0.08

    DEVICE = DEVICE

cfg = Config()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)
print("Config ready.")

## ✅ Step 4 — Dataset  *(Y is NOT loaded — rule change)*
Multi-Z augmentation: randomly sample any Z of same `style_tgt`.
This trains the style encoder to match the **aggregated** style profile
(exactly what the SF leaderboard metric evaluates against).


In [ ]:
class PianoRollDataset(Dataset):
    """
    Returns (X_mix, Z_mix, X_drum, Z_drum) — NO Y.
    Rule change: Y must not be used during training.
    """
    def __init__(self, df, data_root, split="train", augment=False):
        self.df      = df[df["split"] == split].reset_index(drop=True)
        self.root    = Path(data_root)
        self.augment = augment
        # Multi-Z index: any Z of same style_tgt
        self.style_z = defaultdict(list)
        for _, row in df[df["split"] == split].iterrows():
            self.style_z[row["style_tgt"]].append(str(row["Z_path"]))

    def __len__(self): return len(self.df)

    def _load(self, rel_path):
        d = np.load(self.root / rel_path, allow_pickle=True)
        pitched  = d["pitched"].astype(np.float32)  # (n_tracks, 128, 128)
        drum     = d["drum"].astype(np.float32)      # (128, 128)
        track_ids = d["track_ids"] if "track_ids" in d else np.arange(pitched.shape[0])
        return pitched, drum, track_ids

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        X_p, X_d, _ = self._load(row["X_path"])
        # Multi-Z: random Z from same style_tgt
        z_path = random.choice(self.style_z[row["style_tgt"]])
        Z_p, Z_d, _ = self._load(z_path)
        # NO Y LOADING (professor rule change)

        X_mix = X_p.max(0)   # (128,128) mono
        Z_mix = Z_p.max(0)

        # Augmentation: random pitch shift ±2 semitones
        if self.augment and random.random() < 0.5:
            s = random.randint(-2, 2)
            X_mix = np.roll(X_mix, s, axis=0)
            Z_mix = np.roll(Z_mix, s, axis=0)

        return (torch.from_numpy(X_mix.copy()),
                torch.from_numpy(Z_mix.copy()),
                torch.from_numpy(X_d.copy()),
                torch.from_numpy(Z_d.copy()))


splits = df_manifest["split"].unique()
train_ds = PianoRollDataset(df_manifest, cfg.DATA_ROOT, "train", augment=True)
val_ds   = PianoRollDataset(df_manifest, cfg.DATA_ROOT, "val",   augment=False) \
           if "val" in splits else None

train_loader = DataLoader(train_ds, cfg.BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, cfg.BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True) if val_ds else None

print(f"Train: {len(train_ds)} | Val: {len(val_ds) if val_ds else 0} samples")

## ✅ Step 5 — Style Database
Aggregate all training-set Z rolls per `style_tgt`.
This mirrors exactly how SF is evaluated on the leaderboard
(cosine similarity to **aggregated** style profile).


In [ ]:
print("Building aggregated style profiles from training data ...")
style_db = {}

for style_tgt, grp in df_manifest[df_manifest["split"]=="train"].groupby("style_tgt"):
    p_list, d_list = [], []
    for _, row in grp.iterrows():
        d = np.load(cfg.DATA_ROOT / row["Z_path"], allow_pickle=True)
        p_list.append(d["pitched"].astype(np.float32).max(0))
        d_list.append(d["drum"].astype(np.float32))
    style_db[style_tgt] = {
        "pitched": np.mean(p_list, 0),
        "drum":    np.mean(d_list, 0),
    }

print(f"Style DB: {len(style_db)} unique target styles indexed.")

## ✅ Step 6A — Mode A: Deterministic Baseline  *(Submit Immediately!)*
No training needed. Uses velocity histogram matching + Z drum copy.
- **CP ≈ 1.0** (pitches unchanged)
- **SF ≈ 0.40–0.50** (velocity profile matched to Z)
- **HM ≈ 0.57–0.67**

▶ **Run this cell to generate `submission.csv` right now.**


In [ ]:
def quantile_velocity_match(x_roll, z_roll, alpha=cfg.BLEND_ALPHA):
    """Transfer Z's velocity distribution to X's active cells. Pitches unchanged."""
    mask = x_roll > 0
    if mask.sum() == 0: return x_roll.copy()
    z_active = z_roll[z_roll > 0]
    if len(z_active) == 0: return x_roll.copy()
    x_vels   = x_roll[mask]
    z_sorted = np.sort(z_active)
    x_ranks  = np.searchsorted(np.sort(x_vels), x_vels) / len(x_vels)
    z_matched = np.interp(x_ranks, np.linspace(0,1,len(z_sorted),True), z_sorted)
    y = x_roll.copy()
    y[mask] = np.clip((1-alpha)*x_vels + alpha*z_matched, 0, 1)
    return y

def roll_to_notes(roll, threshold=cfg.NOTE_THRESHOLD, min_dur=1):
    """(128,T) float roll → list of (pitch, onset, dur, vel_float)"""
    notes, active = [], {}
    for t in range(roll.shape[1]):
        for p in range(roll.shape[0]):
            v = float(roll[p,t])
            if v > threshold:
                if p not in active: active[p] = (t, v)
            else:
                if p in active:
                    on, ve = active.pop(p)
                    if t-on >= min_dur: notes.append((p, on, t-on, ve))
    for p,(on,ve) in active.items():
        dur = roll.shape[1]-on
        if dur >= min_dur: notes.append((p, on, dur, ve))
    return notes

def build_notes_str(Y_mix, Y_drum, X_pitched, track_ids, drum_tid,
                    threshold=cfg.NOTE_THRESHOLD):
    records = []
    t_scale = np.clip(Y_mix / (X_pitched.max(0) + 1e-8), 0, 3)  # (128,128)
    for tid, x_tr in zip(track_ids, X_pitched):
        y_tr = np.clip(x_tr * t_scale, 0, 1)
        for p,on,dur,vel in roll_to_notes(y_tr, threshold):
            records.append(f"{int(tid)},0,{p},{on},{dur},{max(1,min(127,int(vel*127)))}")
    for p,on,dur,vel in roll_to_notes(Y_drum, threshold):
        records.append(f"{drum_tid},1,{p},{on},{dur},{max(1,min(127,int(vel*127)))}")
    return ";".join(records)

def generate_mode_a(output_path=cfg.SUB_PATH, threshold=cfg.NOTE_THRESHOLD):
    test_df = df_manifest[df_manifest["split"]=="test"]
    if len(test_df)==0:
        print("No test split — using 10 train items as demo.")
        test_df = df_manifest[df_manifest["split"]=="train"].head(10)
    rows = []
    for i,(_, row) in enumerate(test_df.iterrows()):
        Xnpz = np.load(cfg.DATA_ROOT/row["X_path"], allow_pickle=True)
        X_pitched = Xnpz["pitched"].astype(np.float32)
        X_drum    = Xnpz["drum"].astype(np.float32)
        track_ids = Xnpz["track_ids"] if "track_ids" in Xnpz \
                    else np.arange(X_pitched.shape[0])
        drum_tid  = int(Xnpz["drum_track_id"]) if "drum_track_id" in Xnpz \
                    else int(track_ids.max())+1 if len(track_ids)>0 else 9

        style_tgt = row["style_tgt"]
        if style_tgt in style_db:
            Z_mix  = style_db[style_tgt]["pitched"]
            Z_drum = style_db[style_tgt]["drum"]
        else:
            Znpz   = np.load(cfg.DATA_ROOT/row["Z_path"], allow_pickle=True)
            Z_mix  = Znpz["pitched"].astype(np.float32).max(0)
            Z_drum = Znpz["drum"].astype(np.float32)

        X_mix = X_pitched.max(0)
        Y_mix = quantile_velocity_match(X_mix, Z_mix)
        Y_drum = Z_drum.copy()

        notes = build_notes_str(Y_mix, Y_drum, X_pitched, track_ids, drum_tid, threshold)
        rows.append({"item_id": row["item_id"], "notes": notes})

        if (i+1) % 100 == 0: print(f"  {i+1}/{len(test_df)} ...")

    sub = pd.DataFrame(rows, columns=["item_id","notes"])
    sub.to_csv(output_path, index=False)
    print(f"\n[MODE A] Done. {len(sub)} rows → {output_path}")
    return sub

# ── RUN MODE A NOW ──
sub_a = generate_mode_a()
sub_a.head(3)

## ✅ Step 6B — Neural Model: VelocityTransferNet  *(for Mode B)*
Architecture:
- **X encoder** + **Z encoder** (CNN)
- **Cross-attention**: X content tokens attend to Z style tokens
- **vel_scale head**: multiplicative velocity scale for X's notes → `Y = X * scale`
- **drum head**: predict Z-style drum pattern
- CP is structurally guaranteed: `Y = X * scale` never changes pitches.


In [ ]:
class ConvEncoder(nn.Module):
    """(B,1,128,128) → (B,D,16,16)"""
    def __init__(self, D=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,  32,  3,1,1), nn.GELU(),
            nn.Conv2d(32,  64,  4,2,1), nn.GELU(),
            nn.Conv2d(64,  128, 4,2,1), nn.GELU(),
            nn.Conv2d(128, D,   4,2,1), nn.GELU(),
        )
    def forward(self, x): return self.net(x.unsqueeze(1))

class CrossAttnBlock(nn.Module):
    """X content queries attend to Z style keys/values."""
    def __init__(self, D, heads):
        super().__init__()
        self.nq  = nn.LayerNorm(D); self.nkv = nn.LayerNorm(D)
        self.attn = nn.MultiheadAttention(D, heads, batch_first=True)
        self.ff   = nn.Sequential(nn.Linear(D,D*4), nn.GELU(), nn.Linear(D*4,D))
        self.nff  = nn.LayerNorm(D)
    def forward(self, x, z):
        ao, _ = self.attn(self.nq(x), self.nkv(z), self.nkv(z))
        x = x + ao
        return x + self.ff(self.nff(x))

class ConvDecoder(nn.Module):
    """(B,D,16,16) → (B,128,128)"""
    def __init__(self, D=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(D,  128, 4,2,1), nn.GELU(),
            nn.ConvTranspose2d(128, 64, 4,2,1), nn.GELU(),
            nn.ConvTranspose2d(64,   1, 4,2,1),
        )
    def forward(self, x): return self.net(x).squeeze(1)

class VelocityTransferNet(nn.Module):
    """
    Y = X * vel_scale   (CP guaranteed: pitches/onsets unchanged)
    Y_drum = sigmoid(drum_head(Z_features))
    Training uses ONLY X and Z — no Y required.
    """
    def __init__(self, cfg):
        super().__init__()
        D = cfg.HIDDEN
        self.x_enc = ConvEncoder(D)
        self.z_enc = ConvEncoder(D)
        self.sa_z  = nn.TransformerEncoderLayer(D, cfg.N_HEADS, D*4,
                                                 batch_first=True, norm_first=True)
        self.ca    = nn.ModuleList([CrossAttnBlock(D, cfg.N_HEADS)
                                    for _ in range(cfg.N_LAYERS)])
        self.vel_head  = ConvDecoder(D)
        self.drum_head = ConvDecoder(D)
        # Zero-init for stable start
        for m in list(self.vel_head.modules()) + list(self.drum_head.modules()):
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.zeros_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, X_r, Z_r, X_d, Z_d):
        # X_r, Z_r, X_d, Z_d: (B,128,128)
        xf = self.x_enc(X_r)   # (B,D,16,16)
        zf = self.z_enc(Z_r)
        xt = rearrange(xf, "b d h w -> b (h w) d")
        zt = rearrange(zf, "b d h w -> b (h w) d")
        zt = self.sa_z(zt)
        for ca in self.ca: xt = ca(xt, zt)
        xf_out = rearrange(xt, "b (h w) d -> b d h w", h=16, w=16)
        vel_scale = 2.0 * torch.sigmoid(self.vel_head(xf_out))  # (0,2)
        Y_pitched = (X_r * vel_scale).clamp(0, 1)               # CP guaranteed
        Y_drum    = torch.sigmoid(self.drum_head(zf))
        return Y_pitched, Y_drum

model  = VelocityTransferNet(cfg).to(DEVICE)
n_par  = sum(p.numel() for p in model.parameters())/1e6
print(f"VelocityTransferNet | {n_par:.2f}M params")

## ✅ Step 7 — Loss Functions
All losses are **bounded [0,2]** and directly correspond to evaluation metrics:
- `chroma_loss` → maximises **CP** score
- `histogram_loss` → maximises **SF** score (pitch/time marginals)
- `drum_loss` → maximises **SF#4** (onset-drum histogram)

**No reconstruction loss** because there is no Y ground truth.


In [ ]:
def chroma_loss(pred, content):
    """1 - cosine_sim(chroma(pred), chroma(X)). Bounded [0,2]. Optimises CP."""
    def chroma(r):
        return torch.stack([r[:,p::12,:].sum(1) for p in range(12)], 1)
    pc = F.normalize(chroma(pred).flatten(1),    dim=1)
    cc = F.normalize(chroma(content).flatten(1), dim=1)
    return (1.0 - (pc*cc).sum(1)).mean()

def histogram_loss(pred, style):
    """L1 between L1-normalised marginals. Bounded [0,2]. Optimises SF."""
    EPS = 1e-8
    def nl1(a, b):
        an = a / (a.abs().sum(-1,keepdim=True)+EPS)
        bn = b / (b.abs().sum(-1,keepdim=True)+EPS)
        return F.l1_loss(an, bn)
    return (nl1(pred.flatten(1), style.flatten(1))   # joint time-pitch
          + nl1(pred.sum(-1),    style.sum(-1))       # pitch marginal
          + nl1(pred.sum(1),     style.sum(1))        # time marginal
           ) / 3.0

def drum_loss(pred_drum, z_drum):
    """BCE drum pattern loss. Optimises SF#4."""
    target = (z_drum > 0.05).float()
    pw     = torch.tensor(50.0, device=pred_drum.device)
    bce    = F.binary_cross_entropy(pred_drum, target)
    bce_w  = F.binary_cross_entropy_with_logits(pred_drum*2-1, target, pos_weight=pw)
    return bce + 0.5 * bce_w

# Sanity check: all losses should be bounded
with torch.no_grad():
    dummy_X = torch.rand(2,128,128).to(DEVICE)
    dummy_Z = torch.rand(2,128,128).to(DEVICE)
    Yp, Yd = model(dummy_X, dummy_Z, dummy_X, dummy_Z)
    lc = chroma_loss(Yp, dummy_X)
    lh = histogram_loss(Yp, dummy_Z)
    ld = drum_loss(Yd, dummy_Z)
    assert 0 <= float(lc) <= 2.01, f"chroma out of range: {lc}"
    assert 0 <= float(lh) <= 2.01, f"histogram out of range: {lh}"
    print(f"chroma_loss:    {lc:.4f}  [0,2] ✓")
    print(f"histogram_loss: {lh:.4f}  [0,2] ✓")
    print(f"drum_loss:      {ld:.4f}        ✓")
    print("All losses OK.")

## ✅ Step 8 — EMA + Optimiser


In [ ]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay  = decay
        self.shadow = deepcopy(model).eval()
        for p in self.shadow.parameters(): p.requires_grad_(False)
    @torch.no_grad()
    def update(self, m):
        for sp, mp in zip(self.shadow.parameters(), m.parameters()):
            sp.data.mul_(self.decay).add_(mp.data, alpha=1-self.decay)

ema = EMA(model, cfg.EMA_DECAY)

optimizer   = torch.optim.AdamW(model.parameters(), lr=cfg.LR,
                                 weight_decay=cfg.WEIGHT_DECAY)
total_steps = cfg.EPOCHS * len(train_loader)

def lr_lambda(step):
    if step < cfg.WARMUP_STEPS:
        return step / max(1, cfg.WARMUP_STEPS)
    p = (step-cfg.WARMUP_STEPS) / max(1, total_steps-cfg.WARMUP_STEPS)
    return 0.01 + 0.99*0.5*(1+math.cos(math.pi*p))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler(enabled=cfg.AMP)
print(f"Optimiser ready. Total steps: {total_steps}")

## ✅ Step 9 — Training Loop  *(~1 hour on T4)*
Training uses **only X and Z** — no Y ever loaded.
Loss = CP loss + SF histogram loss + drum loss.
Validation computes actual CP and SF metrics every 5 epochs.


In [ ]:
from scipy.spatial.distance import cosine as scipy_cosine

def cp_metric(x, y, stp=4, win=2):
    T = min(x.shape[1], y.shape[1]); sims=[]
    for s in range(0, T-win*stp+1, stp):
        cx = np.stack([x[p::12,s:s+win*stp].sum() for p in range(12)])
        cy = np.stack([y[p::12,s:s+win*stp].sum() for p in range(12)])
        nx,ny = np.linalg.norm(cx),np.linalg.norm(cy)
        if nx>0 and ny>0: sims.append(1-scipy_cosine(cx,cy))
    return float(np.mean(sims)) if sims else 0.0

def sf_metric(zr, zd, yr, yd):
    def sim(a,b):
        a,b=a.flatten(),b.flatten(); na,nb=np.linalg.norm(a),np.linalg.norm(b)
        return (1-scipy_cosine(a/na,b/nb)) if na>0 and nb>0 else 0.0
    return float(np.mean([sim(zr,yr), sim(zr.sum(-1),yr.sum(-1)),
                           sim(zr.sum(0),yr.sum(0)), sim(zd.sum(0),yd.sum(0))]))

@torch.no_grad()
def validate(n=8):
    ema.shadow.eval(); cp_l,sf_l=[],[]
    if val_loader is None: return {}
    for i,(Xr,Zr,Xd,Zd) in enumerate(val_loader):
        if i>=n: break
        Xr,Zr,Xd,Zd = Xr.to(DEVICE),Zr.to(DEVICE),Xd.to(DEVICE),Zd.to(DEVICE)
        Yp,Yd = ema.shadow(Xr,Zr,Xd,Zd)
        for b in range(Xr.shape[0]):
            cp_l.append(cp_metric(Xr[b].cpu().numpy(), Yp[b].cpu().numpy()))
            sf_l.append(sf_metric(Zr[b].cpu().numpy(), Zd[b].cpu().numpy(),
                                   Yp[b].cpu().numpy(), Yd[b].cpu().numpy()))
    cp,sf = float(np.mean(cp_l)), float(np.mean(sf_l))
    hm    = 2*cp*sf/(cp+sf+1e-8)
    return {"CP":cp,"SF":sf,"HM":hm}

def train_one_epoch(epoch):
    model.train(); meters={k:0. for k in ["total","chroma","hist","drum"]}; n=len(train_loader)
    for step,(Xr,Zr,Xd,Zd) in enumerate(train_loader):
        Xr,Zr,Xd,Zd = Xr.to(DEVICE),Zr.to(DEVICE),Xd.to(DEVICE),Zd.to(DEVICE)
        with torch.amp.autocast("cuda", enabled=cfg.AMP):
            Yp,Yd    = model(Xr,Zr,Xd,Zd)
            # Direct metric optimisation — no Y ground truth needed
            lc = chroma_loss(Yp, Xr)          # CP: match X's chroma
            lh = histogram_loss(Yp, Zr)       # SF: match Z's distribution
            ld = drum_loss(Yd, Zd)             # SF#4: match Z's drum
            loss = cfg.W_CHROMA*lc + cfg.W_HIST*lh + cfg.W_DRUM*ld
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.GRAD_CLIP)
        scaler.step(optimizer); scaler.update(); scheduler.step(); ema.update(model)
        for k,v in zip(["total","chroma","hist","drum"],[loss,lc,lh,ld]):
            meters[k]+=v.item()
        if step%100==0:
            lr=optimizer.param_groups[0]["lr"]
            print(f"  E{epoch:03d}[{step:04d}/{n}] loss={loss.item():.4f} "
                  f"chroma={lc.item():.4f} hist={lh.item():.4f} drum={ld.item():.4f} lr={lr:.2e}")
    return {k:v/n for k,v in meters.items()}

def run_training():
    best_hm=-1.; history=[]
    print(f"\nTraining {cfg.EPOCHS} epochs — NO Y used.\n{'='*55}")
    for epoch in range(1, cfg.EPOCHS+1):
        t0=time.time(); m=train_one_epoch(epoch); vm={}
        if val_loader and epoch%5==0:
            vm=validate()
            print(f"  [VAL] CP={vm['CP']:.4f} SF={vm['SF']:.4f} HM={vm['HM']:.4f}")
        history.append({**m,**{f"v_{k}":v for k,v in vm.items()}})
        print(f"[E{epoch:03d}] total={m['total']:.4f} hist={m['hist']:.4f} "
              f"drum={m['drum']:.4f} | {time.time()-t0:.1f}s")
        hm=vm.get("HM",-m["total"])
        if hm>best_hm:
            best_hm=hm
            torch.save({"epoch":epoch,"model":model.state_dict(),
                        "ema":ema.shadow.state_dict(),"best_hm":best_hm},
                       cfg.SAVE_DIR/"best_model.pt")
            print(f"  ✓ Best saved (HM={best_hm:.4f})")
        if epoch%10==0:
            torch.save({"epoch":epoch,"model":model.state_dict(),"ema":ema.shadow.state_dict()},
                       cfg.SAVE_DIR/f"ckpt_ep{epoch:03d}.pt")
    # Plot curves
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    axes[0].plot([h["total"]  for h in history],label="total")
    axes[0].set_title("Total Loss"); axes[0].legend()
    axes[1].plot([h["hist"]   for h in history],color="orange",label="hist→SF")
    axes[1].plot([h["drum"]   for h in history],color="red",   label="drum→SF#4")
    axes[1].plot([h["chroma"] for h in history],color="green", label="chroma→CP")
    axes[1].set_title("Per-metric Losses"); axes[1].legend()
    vhm=[h["v_HM"] for h in history if "v_HM" in h]
    if vhm:
        xs=[5*(i+1) for i in range(len(vhm))]
        axes[2].plot(xs,vhm,color="purple",marker="o",label="Val HM")
        axes[2].axhline(0.65,color="red",ls="--",label="baseline floor")
        axes[2].set_title("Validation HM Score"); axes[2].legend()
    plt.tight_layout(); plt.savefig("/content/training_curves.png",dpi=150); plt.show()
    print(f"Best HM = {best_hm:.4f}")
    return history

# ── UNCOMMENT TO TRAIN ──
# history = run_training()
print("Training function defined. Uncomment last line to start.")

## ✅ Step 10 — Load Checkpoint  *(optional)*
If you already have a saved checkpoint, load it here.


In [ ]:
def load_checkpoint(path="/content/checkpoints/best_model.pt"):
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    ema.shadow.load_state_dict(ckpt["ema"])
    print(f"Loaded epoch={ckpt.get('epoch','?')} | best HM={ckpt.get('best_hm',0):.4f}")

# load_checkpoint()   # uncomment after training

## ✅ Step 11 — Mode B: Neural Submission
Requires a trained model (run Step 9 first or load checkpoint).


In [ ]:
def generate_mode_b(output_path=cfg.SUB_PATH, threshold=cfg.NOTE_THRESHOLD):
    ema.shadow.eval()
    test_df = df_manifest[df_manifest["split"]=="test"]
    if len(test_df)==0:
        print("No test split — using 10 train items as demo.")
        test_df = df_manifest[df_manifest["split"]=="train"].head(10)
    rows=[]; idxs=list(range(len(test_df)))
    for bs in range(0,len(idxs),cfg.INFER_BATCH):
        batch=test_df.iloc[bs:bs+cfg.INFER_BATCH]
        Xb,Zb,Xdb,Zdb,meta=[],[],[],[],[]
        for _,row in batch.iterrows():
            Xnpz=np.load(cfg.DATA_ROOT/row["X_path"],allow_pickle=True)
            X_pitched=Xnpz["pitched"].astype(np.float32)
            X_drum=Xnpz["drum"].astype(np.float32)
            track_ids=Xnpz["track_ids"] if "track_ids" in Xnpz else np.arange(X_pitched.shape[0])
            drum_tid=int(Xnpz["drum_track_id"]) if "drum_track_id" in Xnpz \
                     else int(track_ids.max())+1 if len(track_ids)>0 else 9
            st=row["style_tgt"]
            if st in style_db:
                Z_mix=style_db[st]["pitched"]; Z_drum=style_db[st]["drum"]
            else:
                Znpz=np.load(cfg.DATA_ROOT/row["Z_path"],allow_pickle=True)
                Z_mix=Znpz["pitched"].astype(np.float32).max(0)
                Z_drum=Znpz["drum"].astype(np.float32)
            Xb.append(X_pitched.max(0)); Zb.append(Z_mix)
            Xdb.append(X_drum); Zdb.append(Z_drum)
            meta.append({"item_id":row["item_id"],"X_pitched":X_pitched,
                          "track_ids":track_ids,"drum_tid":drum_tid})
        Xt=torch.from_numpy(np.stack(Xb)).to(DEVICE)
        Zt=torch.from_numpy(np.stack(Zb)).to(DEVICE)
        Xdt=torch.from_numpy(np.stack(Xdb)).to(DEVICE)
        Zdt=torch.from_numpy(np.stack(Zdb)).to(DEVICE)
        with torch.no_grad(): Yp,Yd=ema.shadow(Xt,Zt,Xdt,Zdt)
        for b,m in enumerate(meta):
            notes=build_notes_str(Yp[b].cpu().numpy(), Yd[b].cpu().numpy(),
                                   m["X_pitched"], m["track_ids"], m["drum_tid"], threshold)
            rows.append({"item_id":m["item_id"],"notes":notes})
        if (bs+cfg.INFER_BATCH)%200==0: print(f"  {bs+cfg.INFER_BATCH}/{len(test_df)} ...")
    sub=pd.DataFrame(rows,columns=["item_id","notes"])
    sub.to_csv(output_path,index=False)
    print(f"[MODE B] {len(sub)} rows → {output_path}")
    return sub

# Uncomment after training:
# sub_b = generate_mode_b()
print("Mode B function defined. Train first, then uncomment.")

## 📋 Summary — How to Run

| Step | Cell | Action | Time |
|------|------|--------|------|
| 1 | Install | `!pip install einops` | ~30s |
| 2 | Imports | Run imports & config | instant |
| 3 | Data | Unzip + validate manifest | ~1 min |
| 4 | Dataset | Build loaders (no Y) | instant |
| 5 | Style DB | Aggregate Z profiles | ~2 min |
| **6A** | **Mode A** | **Deterministic submit — run NOW** | **~5 min** |
| 6B | Model | Define VelocityTransferNet | instant |
| 7 | Losses | CP + SF + drum losses | instant |
| 8 | Optim | EMA + AdamW | instant |
| **9** | **Training** | **Uncomment `run_training()`** | **~1 hr** |
| 10 | Checkpoint | Optional: load saved model | instant |
| **11** | **Mode B** | **Uncomment `generate_mode_b()`** | **~5 min** |

### 🏆 Recommended Strategy
1. **Submit Mode A immediately** (baseline, no training needed)
2. **Train Mode B in parallel** — uncomment `run_training()` in Step 9
3. After training finishes, **submit Mode B** for improved score

### 📁 Output file
`/content/submission.csv` — download from Colab file browser (left panel → Files)

### ⚠️ Rule reminder
Y files exist in the dataset but **must NOT be used for training**.
This notebook never loads Y. Training loss = CP loss + SF loss only.
